<a href="https://colab.research.google.com/github/TurboPug2101/Sentiment_Analysis/blob/main/Sentiment_Analysis_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Read the data set
data_url = "/content/amazon_reviews.csv"
sentiment_data = pd.read_csv(data_url)



In [ ]:
from datetime import datetime as dt


In [ ]:
sentiment_data

In [ ]:
data=sentiment_data[['reviewText','overall']]

In [ ]:
data

Vader

In [ ]:
!pip install nltk
import nltk

# Download the lexicon
nltk.download("vader_lexicon")

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Create an instance of SentimentIntensityAnalyzer
sent_analyzer = SentimentIntensityAnalyzer()


In [ ]:
def format_data(data):

    rev_col = str(data.columns[0])
    sc_col = str(data.columns[-1])

    data.rename(columns = {rev_col: 'Review', sc_col: 'polarity'}, inplace=True)

    # Change 0, 2, 4 to negative, neutral and positive
    labels = {1: 'negative',2: 'negative', 3: 'neutral', 4: 'positive',5: 'positive'}
    data['polarity'] = data['polarity'].map(labels)

    # Get only the two columns
    return data[['Review', 'polarity']]

In [ ]:
dataV=format_data(data)

In [ ]:
dataV

In [ ]:
def format_output(output_dict):

    polarity = "neutral"

    if(output_dict['compound']>= 0.05):
        polarity = "positive"

    elif(output_dict['compound']<= -0.05):
        polarity = "negative"

    return polarity

def predict_sentiment(text):

    output_dict =  sent_analyzer.polarity_scores(text)
    return format_output(output_dict)

In [ ]:
dataV = dataV.dropna(subset=["Review"])


In [ ]:
start = dt.now()
dataV["vader_prediction"] = dataV["Review"].apply(predict_sentiment)
running_secs = (dt.now() - start).seconds


In [ ]:
dataV

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(dataV['polarity'], dataV['vader_prediction'])

print("Accuracy: {}\n".format(accuracy))

# Show the classification report
print(classification_report(dataV['polarity'], dataV['vader_prediction']))

In [ ]:
print(running_secs)

# Naive Bayes

In [ ]:
import re
def cleaning(text):
    # converting to lowercase, removing URL links, special characters, punctuations...
    # text = text.lower() # converting to lowercase
    text = re.sub('https?://\S+|www\.\S+', '', text) # removing URL links
    text = re.sub(r"\b\d+\b", "", text) # removing number
    text = re.sub('<.*?>+', '', text) # removing special characters,
    text = re.sub('\n', '', text)
    text = re.sub('[????]', '', text)

    # removing short form:

    text=re.sub("isn't",'is not',text)
    text=re.sub("he's",'he is',text)
    text=re.sub("wasn't",'was not',text)
    text=re.sub("there's",'there is',text)
    text=re.sub("couldn't",'could not',text)
    text=re.sub("won't",'will not',text)
    text=re.sub("they're",'they are',text)
    text=re.sub("she's",'she is',text)
    text=re.sub("There's",'there is',text)
    text=re.sub("wouldn't",'would not',text)
    text=re.sub("haven't",'have not',text)
    text=re.sub("That's",'That is',text)
    text=re.sub("you've",'you have',text)
    text=re.sub("He's",'He is',text)
    text=re.sub("what's",'what is',text)
    text=re.sub("weren't",'were not',text)
    text=re.sub("we're",'we are',text)
    text=re.sub("hasn't",'has not',text)
    text=re.sub("you'd",'you would',text)
    text=re.sub("shouldn't",'should not',text)
    text=re.sub("let's",'let us',text)
    text=re.sub("they've",'they have',text)
    text=re.sub("You'll",'You will',text)
    text=re.sub("i'm",'i am',text)
    text=re.sub("we've",'we have',text)
    text=re.sub("it's",'it is',text)
    text=re.sub("don't",'do not',text)
    text=re.sub("that?s",'that is',text)
    text=re.sub("I?m",'I am',text)
    text=re.sub("it?s",'it is',text)
    text=re.sub("she?s",'she is',text)
    text=re.sub("he?s'",'he is',text)
    text=re.sub('I?m','I am',text)
    text=re.sub('I?d','I did',text)
    text=re.sub("he?s'",'he is',text)
    text=re.sub('there?s','there is',text)


    return text



In [ ]:
dataV = dataV.dropna(subset=["Review"])


In [ ]:
dataN = dataV['Review'].apply(cleaning)

In [ ]:
dataN

In [ ]:
dataN= pd.DataFrame(dataN)

dataN['Sentiment']=dataV['polarity']

In [ ]:
dataN
# dt=dataN

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
from nltk.corpus import stopwords
stop_words = stopwords.words()

dataN['no_sw'] = dataN['Review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))


In [ ]:
dataN

In [ ]:
from collections import Counter
cnt = Counter()
for text in dataN["no_sw"].values:
    for word in text.split():
        cnt[word] += 1
cnt.most_common(10)
temp = pd.DataFrame(cnt.most_common(10))
temp.columns=['word', 'count']
temp

In [ ]:
# Remove the most frequent words:
FREQWORDS = set([w for (w, wc) in cnt.most_common(10)])
def remove_freqwords(text):
    """custom function to remove the frequent words"""
    return " ".join([word for word in str(text).split() if word not in FREQWORDS])
dataN["wo_stopfreq"] = dataN["no_sw"].apply(lambda text: remove_freqwords(text))
dataN.head()

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
wordnet_lem = WordNetLemmatizer()

dataN['wo_stopfreq_lem'] = dataN['wo_stopfreq'].apply(wordnet_lem.lemmatize)
dataN

# Tokenization

In [ ]:
nb=dataN.drop(columns=['Review','no_sw', 'wo_stopfreq'])
nb.columns=['sentiment','review']
nb['sentiment'] = nb['sentiment'].map({'positive': 2, 'negative': 0, 'neutral': 1})
nb


In [ ]:
tokenized_review=nb['review'].apply(lambda x: x.split())
tokenized_review.head(5)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer
token = RegexpTokenizer(r'[a-zA-Z0-9]+')
cv = CountVectorizer(stop_words='english',ngram_range = (1,1),tokenizer = token.tokenize)
text_counts = cv.fit_transform(nb['review'])

TTS


In [ ]:
from sklearn.model_selection import train_test_split
X=text_counts
y=nb['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=30)

Complement NB model:


In [ ]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, confusion_matrix
CNB = ComplementNB()
CNB.fit(X_train, y_train)

from sklearn import metrics

predicted = CNB.predict(X_test)
accuracy_score = metrics.accuracy_score(predicted, y_test)

print('ComplementNB model accuracy is',str('{:04.2f}'.format(accuracy_score*100))+'%')
print('------------------------------------------------')

print('Classification Report:')
print(classification_report(y_test, predicted))

In [ ]:
import time
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report, confusion_matrix
import sklearn.metrics as metrics
start_time = time.time()
CNB = ComplementNB()
CNB.fit(X_train, y_train)
training_time = time.time() - start_time

# Make predictions, measuring the prediction time
start_time = time.time()
predicted = CNB.predict(X_test)
prediction_time = time.time() - start_time

# Calculate accuracy
accuracy_score = metrics.accuracy_score(predicted, y_test)

print('ComplementNB model accuracy is', str('{:04.2f}'.format(accuracy_score * 100)) + '%')
print('Training time:', '{:04.2f}'.format(training_time), 'seconds')
print('Prediction time:', '{:04.2f}'.format(prediction_time), 'seconds')
print('------------------------------------------------')

print('Classification Report:')
print(classification_report(y_test, predicted))

# RNN

In [ ]:
import nltk # Natural Language Toolkit
nltk.download('stopwords')
from nltk.corpus import stopwords
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from numpy import array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding, LSTM, Bidirectional
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [ ]:
dataN

In [ ]:
labelencoder = LabelEncoder()
dataN = dataN.assign(Sentiment = labelencoder.fit_transform(dataN["Sentiment"]))

In [ ]:
docs = dataN['Review']
labels = array(dataN['Sentiment'])

In [ ]:
labels

In [ ]:
X_train, X_test , y_train, y_test = train_test_split(docs, labels , test_size = 0.20)

In [ ]:
X_train

In [ ]:
max_words = 1000  # Maximum number of words to consider
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)
X_train_sequences = tokenizer.texts_to_sequences(X_train)

# Pad sequences so that they have the same length
max_sequence_length = 100  # Adjust this based on your data
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)


In [ ]:
from keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=32, input_length=max_sequence_length))
model.add(SimpleRNN(units=32, activation='relu'))
model.add(Dense(3, activation='softmax'))  # Assuming there are 3 classes

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
start_time = time.time()
model.fit(X_train_padded, y_train, epochs=5, batch_size=64, validation_split=0.2)
end_time = time.time()


In [ ]:
from keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=32, input_length=max_sequence_length))
model.add(SimpleRNN(units=32, activation='relu'))
model.add(Dense(3, activation='softmax'))  # Assuming there are 3 classes

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
start_time = time.time()
model.fit(X_train_padded, y_train, epochs=5, batch_size=64, validation_split=0.2)
end_time = time.time()


In [ ]:
X_test_sequences = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)


In [ ]:
# Predict using the trained model
y_pred = model.predict(X_test_padded)
# Convert predicted probabilities to class labels
y_pred_labels = np.argmax(y_pred, axis=1)
correct_predictions = np.sum(y_pred_labels == y_test)
total_samples = len(y_test)
accuracy = correct_predictions / total_samples
print(f"Test Accuracy: {accuracy:.2f}")

In [ ]:
class_names = ["Negative", "Neutral", "Positive"]  # Update with your actual class names
report = classification_report(y_test, y_pred_labels, target_names=class_names)
print("Classification Report:\n", report)

In [ ]:
prediction_time = end_time - start_time
\

print(f"Prediction Time: {prediction_time:.2f} seconds")